In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, Concatenate, Embedding
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, LeakyReLU
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import datetime
import io
from tensorflow.keras.callbacks import TensorBoard


2026-01-29 12:21:38.467165: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-29 12:21:38.536897: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-29 12:21:40.722664: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
# --- 1. Load and Preprocess Data ---
(x_train, y_train), (_, _) = tf.keras.datasets.cifar10.load_data()
x_train = (x_train.astype('float32') - 127.5) / 127.5

# --- 2. Define Model Constants and Hyperparameters ---
IMG_SHAPE = (32, 32, 3)
NUM_CLASSES = 10
LATENT_DIM = 100
EPOCHS = 30
BATCH_SIZE = 64
SAMPLE_INTERVAL = 2000
LR_D = 0.0004  # Faster learning for the Discriminator
LR_G = 0.0001  # Slower, more stable learning for the Generator
BETA_1 = 0.5




/home/taz/Documents/Masters/DeepLearning/Deep-Learning/DataAugmentationConditionalGan/venv/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


In [ ]:
from tensorflow.keras.layers import BatchNormalization, Dropout, UpSampling2D

def build_generator():
    noise_input = Input(shape=(LATENT_DIM,), name="gen_noise_input")
    label_input = Input(shape=(1,), name="gen_label_input")
    
    # Label embedding
    label_embedding = Embedding(NUM_CLASSES, 50)(label_input)
    label_embedding = Dense(8 * 8)(label_embedding)
    label_embedding = Reshape((8, 8, 1))(label_embedding)
    
    # Noise branch
    noise = Dense(128 * 8 * 8, activation='relu')(noise_input)
    noise = Reshape((8, 8, 128))(noise)
    
    # Combine
    x = Concatenate()([noise, label_embedding])
    
    # Upsampling instead of Transpose to reduce pixelation
    x = UpSampling2D()(x) # 16x16
    x = Conv2D(128, kernel_size=3, padding='same')(x)
    x = BatchNormalization(momentum=0.8)(x)
    x = LeakyReLU(negative_slope=0.2)(x)
    
    x = UpSampling2D()(x) # 32x32
    x = Conv2D(64, kernel_size=3, padding='same')(x)
    x = BatchNormalization(momentum=0.8)(x)
    x = LeakyReLU(negative_slope=0.2)(x)
    
    x = Conv2D(3, kernel_size=3, padding='same', activation='tanh')(x)
    
    return Model([noise_input, label_input], x, name="generator")

# The original research emphasizes this specific initializer
init = tf.keras.initializers.RandomNormal(stddev=0.02)

def build_discriminator():
    # Image Branch
    img_input = Input(shape=IMG_SHAPE, name="disc_img_input")
    
    # Label Branch - Better to keep the embedding dense until later 
    # OR ensure it is initialized correctly
    label_input = Input(shape=(1,), name="disc_label_input")
    label_embedding = Embedding(NUM_CLASSES, 50)(label_input)
    label_embedding = Dense(IMG_SHAPE[0] * IMG_SHAPE[1], kernel_initializer=init)(label_embedding)
    label_embedding = Reshape((IMG_SHAPE[0], IMG_SHAPE[1], 1))(label_embedding)
    
    # Concatenate
    x = Concatenate()([img_input, label_embedding])
    
    # Layer 1: 32x32 -> 16x16
    x = Conv2D(64, kernel_size=4, strides=2, padding='same', kernel_initializer=init)(x)
    x = LeakyReLU(alpha=0.2)(x)
    # Research suggests NO Dropout in the first layer of D
    
    # Layer 2: 16x16 -> 8x8
    x = Conv2D(128, kernel_size=4, strides=2, padding='same', kernel_initializer=init)(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dropout(0.3)(x) 
    
    # Layer 3: 8x8 -> 4x4
    x = Conv2D(256, kernel_size=4, strides=2, padding='same', kernel_initializer=init)(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dropout(0.3)(x)

    x = Flatten()(x)
    # The output MUST use the same initializer to avoid gradient explosion
    x = Dense(1, activation='sigmoid', kernel_initializer=init)(x)
    
    return Model([img_input, label_input], x, name="discriminator")

In [ ]:
# --- 4. Build and Compile Models ---
optimizer = Adam(learning_rate=LR_D, beta_1=BETA_1)
discriminator = build_discriminator()
discriminator.compile(loss='mean_squared_error', optimizer=optimizer, metrics=['accuracy'])

optimizer_g = Adam(learning_rate=LR_G, beta_1=BETA_1)
generator = build_generator()   
discriminator.trainable = False

noise_input = Input(shape=(LATENT_DIM,))
label_input = Input(shape=(1,))
generated_img = generator([noise_input, label_input])
validity = discriminator([generated_img, label_input])

cgan = Model([noise_input, label_input], validity, name="cgan")
cgan.compile(loss='mean_squared_error', optimizer=optimizer_g)



2026-01-29 12:21:52.007190: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [ ]:
# --- 5. Image Sampling and Logging Function ---
def sample_and_log_images(epoch, generator, summary_writer, num_samples=10):
    """Generate sample images and log them to TensorBoard."""
    # Generate images for each class (0-9)
    noise = np.random.normal(0, 1, (num_samples, LATENT_DIM))
    sampled_labels = np.arange(0, num_samples).reshape(-1, 1)
    
    # Generate images
    gen_imgs = generator.predict([noise, sampled_labels], verbose=0)
    
    # Rescale images from [-1, 1] to [0, 1]
    gen_imgs = 0.5 * gen_imgs + 0.5
    gen_imgs = np.clip(gen_imgs, 0, 1)
    
    # Log to TensorBoard
    with summary_writer.as_default():
        tf.summary.image(
            "Generated Images", 
            gen_imgs, 
            max_outputs=num_samples, 
            step=epoch
        )

In [6]:
def train_cgan(epochs, batch_size, sample_interval):
    log_dir = "logs/adversarial/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    summary_writer = tf.summary.create_file_writer(log_dir)
    steps_per_epoch = x_train.shape[0] // batch_size

    for epoch in range(epochs):
        # Shuffle for "True Epoch" logic
        idx_shuffle = np.random.permutation(x_train.shape[0])
        x_tr, y_tr = x_train[idx_shuffle], y_train[idx_shuffle]
        
        for step in range(steps_per_epoch):
            # 1. Prepare Data
            idx = slice(step * batch_size, (step + 1) * batch_size)
            real_imgs, labels = x_tr[idx], y_tr[idx]
            
            # --- Label Smoothing ---
            valid = np.ones((batch_size, 1)) * 0.9 # Soft labels for real
            fake = np.zeros((batch_size, 1))      # Hard labels for fake
            
            # 2. Train Discriminator
            noise = np.random.normal(0, 1, (batch_size, LATENT_DIM))
            gen_imgs = generator.predict([noise, labels], verbose=0)
            
            d_loss_real = discriminator.train_on_batch([real_imgs, labels], valid)
            d_loss_fake = discriminator.train_on_batch([gen_imgs, labels], fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # 3. Train Generator
            # Note: We use hard 1.0 labels for the generator to maximize "fooling"
            g_loss = cgan.train_on_batch([noise, labels], np.ones((batch_size, 1)))

        # Log to TensorBoard at the end of each epoch
        with summary_writer.as_default():
            tf.summary.scalar('d_loss', d_loss[0], step=epoch)
            tf.summary.scalar('d_accuracy', d_loss[1], step=epoch)
            tf.summary.scalar('g_loss', g_loss, step=epoch)
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch} [D Loss: {d_loss[0]:.4f}, Acc: {100*d_loss[1]:.2f}%] [G Loss: {g_loss:.4f}]")
            sample_and_log_images(epoch, generator, summary_writer)

In [18]:
# --- 6. Start Training ---
#train_cgan(epochs=EPOCHS, batch_size=BATCH_SIZE, sample_interval=SAMPLE_INTERVAL)

# --- 7. Launch TensorBoard ---
%reload_ext tensorboard
%tensorboard --logdir /home/taz/Documents/Masters/DeepLearning/Deep-Learning/DataAugmentationConditionalGan/kaggle_outputs/output_adversarial_stress_3/

Reusing TensorBoard on port 6010 (pid 53219), started 0:00:04 ago. (Use '!kill 53219' to kill it.)

In [ ]:
# # --- 8. Save Models for Download ---
# import shutil
# import os

# print("--- Saving Trained Models ---")

# # Define the folder where models will be saved
# MODEL_SAVE_PATH = '/kaggle/working/cgan_models'

# # Create directory if it doesn't exist
# os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

# # Save generator and discriminator models
# generator.save(os.path.join(MODEL_SAVE_PATH, 'generator.keras'))
# discriminator.save(os.path.join(MODEL_SAVE_PATH, 'discriminator.keras'))

# print(f"Generator saved to {MODEL_SAVE_PATH}/generator.keras")
# print(f"Discriminator saved to {MODEL_SAVE_PATH}/discriminator.keras")

# # Zip the folder for easy download
# shutil.make_archive('/kaggle/working/cgan_models_bundle', 'zip', MODEL_SAVE_PATH)

# print("\n DONE! Download 'cgan_models_bundle.zip' from the Output files on the right.")